# 03 — Final KoVAE Synthetic Generation

This notebook is updated for the new main KoVAE model trained in Notebook 02.

The new KoVAE is stronger internally because it uses:

```text
activity embeddings
subject embeddings
activity classifier loss
activity-conditioned Koopman matrices
```

So for generation, we do **not** use the extra quality filter by default.

Final methods:

```text
rollout_v1
posterior_bank_v2
```

Meaning:

```text
rollout_v1        = activity-conditioned Koopman rollout baseline
posterior_bank_v2 = activity-conditioned posterior-bank generation
```

Final output folders:

```text
data/synthetic_subjects/kovae/rollout_v1/
data/synthetic_subjects/kovae/posterior_bank_v2/

results/kovae_generation/rollout_v1/
results/kovae_generation/posterior_bank_v2/

figures/kovae_generation/rollout_v1/
figures/kovae_generation/posterior_bank_v2/
```

Notebook 04 and Notebook 05 can keep using the same method names.


## Fix included

This version fixes the checkpoint loading error by matching the Notebook 02 encoder architecture exactly:

```text
Conv1d -> GELU -> BatchNorm1d -> Conv1d -> GELU -> BatchNorm1d -> Dropout
```

The previous generation notebook had a different encoder definition, so `load_state_dict()` failed.


In [1]:

# ============================================================
# 03_generate_kovae_synthetic_subjects_final.py
#
# Final synthetic generation notebook for the updated main KoVAE.
#
# This generation code matches the updated Notebook 02 model:
#   - activity embeddings
#   - subject embeddings
#   - activity classifier loss during training
#   - learned activity-conditioned Koopman matrices
#
# No quality filter is used by default.
#
# Final methods:
#   1) rollout_v1
#   2) posterior_bank_v2
#
# Output folders:
#   data/synthetic_subjects/kovae/<method>/
#   results/kovae_generation/<method>/
#   figures/kovae_generation/<method>/
# ============================================================

from __future__ import annotations

import json
import math
import random
import warnings
from dataclasses import asdict, dataclass, field, fields
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Config
# ============================================================

@dataclass
class KoVAEConfig:
    project_root: str = "/home/iailab42/khans1/projects/ir"
    processed_subdir: str = "data/processed/native_rates"

    experiment_name: str = "kovae"

    random_seed: int = 42

    train_subjects: List[str] = field(default_factory=lambda: [
        "S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"
    ])
    val_subjects: List[str] = field(default_factory=lambda: ["S14", "S15"])
    test_subjects: List[str] = field(default_factory=lambda: ["S7", "S8", "S10"])

    acc_hz: int = 32
    bvp_hz: int = 64
    slow_hz: int = 4

    acc_len: int = 256
    bvp_len: int = 512
    slow_len: int = 32

    acc_channels: int = 3
    bvp_channels: int = 1
    slow_channels: int = 2

    latent_steps: int = 64
    branch_hidden_dim: int = 64
    fusion_hidden_dim: int = 128
    latent_dim: int = 32

    activity_embedding_dim: int = 16
    use_subject_condition: bool = True
    subject_embedding_dim: int = 16
    subject_dropout_prob: float = 0.20

    activity_classifier_weight: float = 0.20
    use_activity_conditioned_koopman: bool = True
    activity_koopman_noise_std: float = 0.01
    activity_koopman_identity_regularization: float = 1e-5

    dropout: float = 0.10

    batch_size: int = 64
    num_workers: int = 0
    epochs: int = 100
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    patience: int = 15
    gradient_clip_norm: float = 1.0

    use_weighted_sampler: bool = True
    use_amp: bool = True

    reconstruction_weight_bvp: float = 1.0
    reconstruction_weight_acc: float = 1.0
    reconstruction_weight_slow: float = 1.0

    alpha_koopman: float = 0.10
    beta_kl: float = 1e-3
    koopman_ridge: float = 1e-3

    max_reconstruction_examples: int = 3


@dataclass
class GenerationConfig:
    project_root: str = "/home/iailab42/khans1/projects/ir"

    processed_subdir: str = "data/processed/native_rates"

    checkpoint_subdir: str = "models/checkpoints"
    checkpoint_filename: str = "kovae_best.pt"

    kovae_results_subdir: str = "results/kovae"

    synthetic_base_subdir: str = "data/synthetic_subjects/kovae"
    results_base_subdir: str = "results/kovae_generation"
    figures_base_subdir: str = "figures/kovae_generation"
    configs_subdir: str = "configs"

    methods_to_run: List[str] = field(default_factory=lambda: [
        "rollout_v1",
        "posterior_bank_v2",
    ])

    random_seed: int = 42

    num_synthetic_subjects: int = 10
    windows_per_subject: int = 3000
    generation_batch_size: int = 128

    # Activity sampling:
    #   "train_distribution" = preserve real train label distribution
    #   "balanced" = uniform over activities
    activity_sampling_mode: str = "train_distribution"

    # Subject conditioning:
    #   "sample_train_subject" = assign each synthetic subject one train subject token
    #   "UNK" = use the robust unknown subject token for all synthetic subjects
    synthetic_subject_condition_mode: str = "sample_train_subject"

    # Stable activity-specific Koopman matrices are used for generation.
    target_spectral_radius: float = 0.98

    # rollout_v1 parameters
    rollout_z0_noise_scale: float = 0.05
    rollout_latent_noise_scale: float = 0.03

    # posterior_bank_v2 parameters
    posterior_noise_scale: float = 0.04
    posterior_interpolation_prob: float = 0.50
    posterior_koopman_blend_weight: float = 0.10

    save_plot: bool = True
    num_example_plots: int = 3


GEN_CONFIG = GenerationConfig()


# ============================================================
# Paths and reproducibility
# ============================================================

def get_paths(config: GenerationConfig) -> Dict[str, Path]:
    root = Path(config.project_root)

    return {
        "root": root,
        "processed": root / config.processed_subdir,
        "checkpoint": root / config.checkpoint_subdir / config.checkpoint_filename,
        "kovae_results": root / config.kovae_results_subdir,
        "synthetic_base": root / config.synthetic_base_subdir,
        "results_base": root / config.results_base_subdir,
        "figures_base": root / config.figures_base_subdir,
        "configs": root / config.configs_subdir,
    }


def get_method_paths(base_paths: Dict[str, Path], method_name: str) -> Dict[str, Path]:
    return {
        "synthetic": base_paths["synthetic_base"] / method_name,
        "results": base_paths["results_base"] / method_name,
        "figures": base_paths["figures_base"] / method_name,
    }


def make_dirs(*paths: Path) -> None:
    for path in paths:
        path.mkdir(parents=True, exist_ok=True)


def save_json(data: Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def config_from_checkpoint_dict(config_dict: Dict) -> KoVAEConfig:
    valid_fields = {f.name for f in fields(KoVAEConfig)}
    clean = {k: v for k, v in config_dict.items() if k in valid_fields}

    return KoVAEConfig(**clean)


def load_torch_checkpoint(path: Path, device: torch.device) -> Dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing KoVAE checkpoint: {path}")

    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


# ============================================================
# Model definitions copied from Notebook 02
# ============================================================

class BranchEncoder(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_dim: int,
        latent_steps: int,
        dropout: float,
    ) -> None:
        super().__init__()

        # Must match Notebook 02 exactly.
        # The trained checkpoint contains BatchNorm1d layers at net.2 and net.5.
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=7, padding=3),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(latent_steps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        h = self.net(x)
        h = self.pool(h)
        return h.transpose(1, 2)


class BranchDecoder(nn.Module):
    def __init__(
        self,
        latent_dim: int,
        condition_dim: int,
        hidden_dim: int,
        output_len: int,
        output_channels: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.output_len = output_len

        self.gru = nn.GRU(
            input_size=latent_dim + condition_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )

        self.conv = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, output_channels, kernel_size=3, padding=1),
        )

    def forward(self, z: torch.Tensor, condition_seq: torch.Tensor) -> torch.Tensor:
        h = torch.cat([z, condition_seq], dim=-1)
        h, _ = self.gru(h)
        h = h.transpose(1, 2)
        h = F.interpolate(h, size=self.output_len, mode="linear", align_corners=False)
        out = self.conv(h)
        return out.transpose(1, 2)


class MultiBranchKoVAE(nn.Module):
    def __init__(
        self,
        config: KoVAEConfig,
        num_activities: int,
        num_subject_tokens: int,
    ) -> None:
        super().__init__()

        self.config = config
        self.num_activities = num_activities
        self.num_subject_tokens = num_subject_tokens

        self.bvp_encoder = BranchEncoder(
            in_channels=config.bvp_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )
        self.acc_encoder = BranchEncoder(
            in_channels=config.acc_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )
        self.slow_encoder = BranchEncoder(
            in_channels=config.slow_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )

        self.activity_embedding = nn.Embedding(num_activities, config.activity_embedding_dim)

        if config.use_subject_condition:
            self.subject_embedding = nn.Embedding(num_subject_tokens, config.subject_embedding_dim)
            condition_dim = config.activity_embedding_dim + config.subject_embedding_dim
        else:
            self.subject_embedding = None
            condition_dim = config.activity_embedding_dim

        self.condition_dim = condition_dim

        fusion_input_dim = 3 * config.branch_hidden_dim + condition_dim

        self.fusion_gru = nn.GRU(
            input_size=fusion_input_dim,
            hidden_size=config.fusion_hidden_dim,
            batch_first=True,
        )

        self.to_mu = nn.Linear(config.fusion_hidden_dim, config.latent_dim)
        self.to_logvar = nn.Linear(config.fusion_hidden_dim, config.latent_dim)

        self.activity_classifier = nn.Sequential(
            nn.LayerNorm(config.latent_dim),
            nn.Linear(config.latent_dim, config.fusion_hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.fusion_hidden_dim // 2, num_activities),
        )

        self.activity_koopman = nn.Parameter(
            torch.empty(num_activities, config.latent_dim, config.latent_dim)
        )
        self.reset_activity_koopman_parameters()

        self.bvp_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.bvp_len,
            output_channels=config.bvp_channels,
            dropout=config.dropout,
        )
        self.acc_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.acc_len,
            output_channels=config.acc_channels,
            dropout=config.dropout,
        )
        self.slow_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.slow_len,
            output_channels=config.slow_channels,
            dropout=config.dropout,
        )

    def reset_activity_koopman_parameters(self) -> None:
        with torch.no_grad():
            eye = torch.eye(self.config.latent_dim)
            eye = eye[None, :, :].repeat(self.num_activities, 1, 1)
            noise = self.config.activity_koopman_noise_std * torch.randn_like(eye)
            self.activity_koopman.copy_(eye + noise)

    def make_condition(self, activity: torch.Tensor, subject: torch.Tensor) -> torch.Tensor:
        activity_emb = self.activity_embedding(activity)

        if self.config.use_subject_condition:
            subject_emb = self.subject_embedding(subject)
            condition = torch.cat([activity_emb, subject_emb], dim=-1)
        else:
            condition = activity_emb

        return condition

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(
        self,
        bvp: torch.Tensor,
        acc: torch.Tensor,
        slow: torch.Tensor,
        activity: torch.Tensor,
        subject: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        bvp_h = self.bvp_encoder(bvp)
        acc_h = self.acc_encoder(acc)
        slow_h = self.slow_encoder(slow)

        condition = self.make_condition(activity, subject)
        condition_seq = condition[:, None, :].repeat(1, self.config.latent_steps, 1)

        fused = torch.cat([bvp_h, acc_h, slow_h, condition_seq], dim=-1)
        fused_h, _ = self.fusion_gru(fused)

        mu = self.to_mu(fused_h)
        logvar = self.to_logvar(fused_h).clamp(min=-8.0, max=8.0)

        z = self.reparameterize(mu, logvar)

        recon_bvp = self.bvp_decoder(z, condition_seq)
        recon_acc = self.acc_decoder(z, condition_seq)
        recon_slow = self.slow_decoder(z, condition_seq)

        pooled_mu = mu.mean(dim=1)
        activity_logits = self.activity_classifier(pooled_mu)

        return {
            "recon_bvp": recon_bvp,
            "recon_acc": recon_acc,
            "recon_slow": recon_slow,
            "mu": mu,
            "logvar": logvar,
            "z": z,
            "condition": condition,
            "activity_logits": activity_logits,
        }

    def get_activity_koopman_matrices(self) -> torch.Tensor:
        return self.activity_koopman


# ============================================================
# Data helpers
# ============================================================

def load_native_rate_arrays(processed_dir: Path) -> Dict[str, np.ndarray]:
    required_files = {
        "X_acc": "all_X_acc_32hz.npy",
        "X_bvp": "all_X_bvp_64hz.npy",
        "X_slow": "all_X_slow_4hz.npy",
        "y": "all_y.npy",
        "subjects": "all_subject.npy",
    }

    arrays = {}
    missing = []

    for key, filename in required_files.items():
        path = processed_dir / filename

        if not path.exists():
            missing.append(str(path))
        else:
            if key == "subjects":
                arrays[key] = np.load(path, allow_pickle=True).astype(str)
            else:
                arrays[key] = np.load(path)

    if missing:
        raise FileNotFoundError(
            "Missing required preprocessing files:\n" + "\n".join(missing)
        )

    return arrays


def validate_arrays(arrays: Dict[str, np.ndarray], config: KoVAEConfig) -> None:
    n = len(arrays["y"])

    expected_shapes = {
        "X_acc": (n, config.acc_len, config.acc_channels),
        "X_bvp": (n, config.bvp_len, config.bvp_channels),
        "X_slow": (n, config.slow_len, config.slow_channels),
    }

    for key, expected in expected_shapes.items():
        if arrays[key].shape != expected:
            raise ValueError(f"{key}: expected {expected}, got {arrays[key].shape}")

        if not np.all(np.isfinite(arrays[key])):
            raise ValueError(f"{key} contains NaN or infinite values.")

    if len(arrays["subjects"]) != n:
        raise ValueError("subjects length does not match y length.")


def build_indices_by_subject(subjects: np.ndarray, selected_subjects: List[str]) -> np.ndarray:
    mask = np.isin(subjects.astype(str), np.asarray(selected_subjects, dtype=str))
    return np.where(mask)[0].astype(np.int64)


def encode_activities(y: np.ndarray, mapping: Dict[str, int]) -> np.ndarray:
    encoded = []

    for label in y:
        key = str(int(label))
        if key not in mapping:
            raise KeyError(f"Activity label {key} not found in mapping.")
        encoded.append(mapping[key])

    return np.asarray(encoded, dtype=np.int64)


def encode_subjects(subjects: np.ndarray, mapping: Dict[str, int]) -> np.ndarray:
    return np.asarray([mapping.get(str(s), mapping["UNK"]) for s in subjects], dtype=np.int64)


def make_idx_to_activity_label(activity_to_idx: Dict[str, int]) -> Dict[int, int]:
    return {int(idx): int(label) for label, idx in activity_to_idx.items()}


def make_idx_to_subject(subject_to_idx: Dict[str, int]) -> Dict[int, str]:
    return {int(idx): str(subject) for subject, idx in subject_to_idx.items()}


# ============================================================
# Model/checkpoint loading
# ============================================================

def load_model_and_context(
    gen_config: GenerationConfig,
    device: torch.device,
) -> Tuple[MultiBranchKoVAE, KoVAEConfig, Dict[str, int], Dict[str, int], Dict[str, Path]]:
    paths = get_paths(gen_config)

    checkpoint = load_torch_checkpoint(paths["checkpoint"], device=device)

    if "config" not in checkpoint:
        raise KeyError("Checkpoint does not contain a config dictionary.")

    train_config = config_from_checkpoint_dict(checkpoint["config"])

    activity_to_idx = {
        str(key): int(value)
        for key, value in checkpoint["activity_to_idx"].items()
    }
    subject_to_idx = {
        str(key): int(value)
        for key, value in checkpoint["subject_to_idx"].items()
    }

    model = MultiBranchKoVAE(
        config=train_config,
        num_activities=len(activity_to_idx),
        num_subject_tokens=len(subject_to_idx),
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()

    return model, train_config, activity_to_idx, subject_to_idx, paths


# ============================================================
# Koopman helpers
# ============================================================

def spectral_radius(matrix: np.ndarray) -> float:
    eigvals = np.linalg.eigvals(matrix.astype(np.float64))
    return float(np.max(np.abs(eigvals)))


def stabilize_activity_koopman_matrices(
    activity_A_raw: np.ndarray,
    target_radius: float,
) -> Tuple[np.ndarray, pd.DataFrame]:
    """
    Scale each activity-specific Koopman matrix if its spectral radius is
    larger than target_radius.
    """

    activity_A_raw = np.asarray(activity_A_raw, dtype=np.float32)
    activity_A_stable = activity_A_raw.copy()

    rows = []

    for activity_idx in range(activity_A_raw.shape[0]):
        A = activity_A_raw[activity_idx]
        radius = spectral_radius(A)

        if radius > target_radius and radius > 0:
            scale = target_radius / radius
        else:
            scale = 1.0

        activity_A_stable[activity_idx] = A * scale
        stable_radius = spectral_radius(activity_A_stable[activity_idx])

        rows.append(
            {
                "activity_idx": int(activity_idx),
                "raw_spectral_radius": float(radius),
                "target_spectral_radius": float(target_radius),
                "scaling_factor": float(scale),
                "stable_spectral_radius": float(stable_radius),
            }
        )

    summary_df = pd.DataFrame(rows)

    return activity_A_stable.astype(np.float32), summary_df


# ============================================================
# Latent bank collection
# ============================================================

@torch.no_grad()
def collect_activity_latent_banks(
    model: MultiBranchKoVAE,
    arrays: Dict[str, np.ndarray],
    y_encoded: np.ndarray,
    subject_encoded: np.ndarray,
    train_indices: np.ndarray,
    batch_size: int,
    device: torch.device,
) -> Dict[str, Dict[int, np.ndarray]]:
    """
    Encodes real train windows and stores activity-specific posterior means.

    z0_banks:
        first latent state per train window, used by rollout_v1

    trajectory_banks:
        full posterior latent trajectories, used by posterior_bank_v2
    """

    model.eval()

    z0_banks = {}
    trajectory_banks = {}

    for start in range(0, len(train_indices), batch_size):
        end = min(start + batch_size, len(train_indices))
        idx = train_indices[start:end]

        acc = torch.from_numpy(arrays["X_acc"][idx]).float().to(device)
        bvp = torch.from_numpy(arrays["X_bvp"][idx]).float().to(device)
        slow = torch.from_numpy(arrays["X_slow"][idx]).float().to(device)
        activity = torch.from_numpy(y_encoded[idx]).long().to(device)
        subject = torch.from_numpy(subject_encoded[idx]).long().to(device)

        outputs = model(
            bvp=bvp,
            acc=acc,
            slow=slow,
            activity=activity,
            subject=subject,
        )

        mu = outputs["mu"].detach().cpu().numpy().astype(np.float32)
        activity_np = activity.detach().cpu().numpy().astype(np.int64)

        for local_i, activity_idx in enumerate(activity_np):
            activity_idx = int(activity_idx)

            if activity_idx not in z0_banks:
                z0_banks[activity_idx] = []
                trajectory_banks[activity_idx] = []

            z0_banks[activity_idx].append(mu[local_i, 0, :])
            trajectory_banks[activity_idx].append(mu[local_i])

    z0_banks = {
        key: np.stack(value, axis=0).astype(np.float32)
        for key, value in z0_banks.items()
    }
    trajectory_banks = {
        key: np.stack(value, axis=0).astype(np.float32)
        for key, value in trajectory_banks.items()
    }

    print("\nLatent bank sizes:")
    for activity_idx in sorted(trajectory_banks):
        print(
            f"activity_idx={activity_idx} | "
            f"z0={z0_banks[activity_idx].shape} | "
            f"trajectory={trajectory_banks[activity_idx].shape}"
        )

    return {
        "z0_banks": z0_banks,
        "trajectory_banks": trajectory_banks,
    }


def save_latent_banks(
    banks: Dict[str, Dict[int, np.ndarray]],
    output_dir: Path,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    for bank_name, bank in banks.items():
        save_dict = {
            f"activity_{activity_idx}": value
            for activity_idx, value in bank.items()
        }
        np.savez_compressed(output_dir / f"shared_{bank_name}.npz", **save_dict)


# ============================================================
# Generation helpers
# ============================================================

def make_activity_probability(
    y_train_encoded: np.ndarray,
    num_activities: int,
    mode: str,
) -> np.ndarray:
    if mode == "balanced":
        return np.ones(num_activities, dtype=np.float64) / float(num_activities)

    if mode == "train_distribution":
        counts = np.bincount(y_train_encoded, minlength=num_activities).astype(np.float64)
        counts = np.maximum(counts, 1.0)
        return counts / counts.sum()

    raise ValueError("activity_sampling_mode must be 'balanced' or 'train_distribution'.")


def create_synthetic_activity_sequence(
    num_windows: int,
    y_train_encoded: np.ndarray,
    num_activities: int,
    gen_config: GenerationConfig,
    rng: np.random.Generator,
) -> np.ndarray:
    probabilities = make_activity_probability(
        y_train_encoded=y_train_encoded,
        num_activities=num_activities,
        mode=gen_config.activity_sampling_mode,
    )

    return rng.choice(
        np.arange(num_activities, dtype=np.int64),
        size=num_windows,
        replace=True,
        p=probabilities,
    ).astype(np.int64)


def create_synthetic_subject_names(gen_config: GenerationConfig) -> np.ndarray:
    names = []

    for subject_idx in range(gen_config.num_synthetic_subjects):
        subject_name = f"synthetic_subject_{subject_idx + 1:02d}"
        names.extend([subject_name] * gen_config.windows_per_subject)

    return np.asarray(names, dtype=object)


def assign_subject_condition_tokens(
    synthetic_subjects: np.ndarray,
    subject_to_idx: Dict[str, int],
    gen_config: GenerationConfig,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, Dict[str, str]]:
    """
    Assign subject condition tokens for decoding.

    sample_train_subject:
        each synthetic subject gets one randomly sampled train subject token.

    UNK:
        all synthetic subjects use the UNK token.
    """

    unique_syn_subjects = sorted(np.unique(synthetic_subjects.astype(str)).tolist())
    unk_idx = int(subject_to_idx["UNK"])

    train_subject_names = [
        subject
        for subject in subject_to_idx
        if subject != "UNK"
    ]

    if len(train_subject_names) == 0:
        mode = "UNK"
    else:
        mode = gen_config.synthetic_subject_condition_mode

    mapping = {}

    for syn_subject in unique_syn_subjects:
        if mode == "UNK":
            mapping[syn_subject] = "UNK"

        elif mode == "sample_train_subject":
            mapping[syn_subject] = str(rng.choice(train_subject_names))

        else:
            raise ValueError(
                "synthetic_subject_condition_mode must be 'sample_train_subject' or 'UNK'."
            )

    subject_tokens = np.asarray(
        [
            int(subject_to_idx.get(mapping[str(subject)], unk_idx))
            for subject in synthetic_subjects.astype(str)
        ],
        dtype=np.int64,
    )

    return subject_tokens, mapping


def generate_latent_rollout_batch(
    activity_batch: np.ndarray,
    z0_banks: Dict[int, np.ndarray],
    activity_A_stable: np.ndarray,
    train_config: KoVAEConfig,
    gen_config: GenerationConfig,
    rng: np.random.Generator,
) -> np.ndarray:
    batch_size = len(activity_batch)
    latent_steps = train_config.latent_steps
    latent_dim = train_config.latent_dim

    z = np.zeros((batch_size, latent_steps, latent_dim), dtype=np.float32)

    for i, activity_idx in enumerate(activity_batch):
        activity_idx = int(activity_idx)

        bank = z0_banks[activity_idx]
        z0 = bank[rng.integers(0, len(bank))].copy()

        z0 += rng.normal(
            loc=0.0,
            scale=gen_config.rollout_z0_noise_scale,
            size=z0.shape,
        ).astype(np.float32)

        z[i, 0] = z0

        A = activity_A_stable[activity_idx].astype(np.float32)

        for t in range(1, latent_steps):
            noise = rng.normal(
                loc=0.0,
                scale=gen_config.rollout_latent_noise_scale,
                size=(latent_dim,),
            ).astype(np.float32)

            z[i, t] = z[i, t - 1] @ A + noise

    return z.astype(np.float32)


def generate_posterior_bank_batch(
    activity_batch: np.ndarray,
    trajectory_banks: Dict[int, np.ndarray],
    activity_A_stable: np.ndarray,
    gen_config: GenerationConfig,
    rng: np.random.Generator,
) -> np.ndarray:
    trajectories = []

    for activity_idx in activity_batch:
        activity_idx = int(activity_idx)
        bank = trajectory_banks[activity_idx]

        z = bank[rng.integers(0, len(bank))].copy()

        if rng.random() < gen_config.posterior_interpolation_prob and len(bank) > 1:
            z2 = bank[rng.integers(0, len(bank))].copy()
            lam = rng.uniform(0.25, 0.75)
            z = lam * z + (1.0 - lam) * z2

        z += rng.normal(
            loc=0.0,
            scale=gen_config.posterior_noise_scale,
            size=z.shape,
        ).astype(np.float32)

        blend = float(gen_config.posterior_koopman_blend_weight)

        if blend > 0:
            A = activity_A_stable[activity_idx].astype(np.float32)
            z_pred_next = z[:-1] @ A
            z[1:] = (1.0 - blend) * z[1:] + blend * z_pred_next

        trajectories.append(z.astype(np.float32))

    return np.stack(trajectories, axis=0).astype(np.float32)


@torch.no_grad()
def decode_latent_batch(
    model: MultiBranchKoVAE,
    z_np: np.ndarray,
    activity_batch: np.ndarray,
    subject_token_batch: np.ndarray,
    device: torch.device,
) -> Dict[str, np.ndarray]:
    model.eval()

    z = torch.from_numpy(z_np).float().to(device)
    activity = torch.from_numpy(activity_batch.astype(np.int64)).long().to(device)
    subject = torch.from_numpy(subject_token_batch.astype(np.int64)).long().to(device)

    condition = model.make_condition(activity, subject)
    condition_seq = condition[:, None, :].repeat(1, model.config.latent_steps, 1)

    recon_bvp = model.bvp_decoder(z, condition_seq)
    recon_acc = model.acc_decoder(z, condition_seq)
    recon_slow = model.slow_decoder(z, condition_seq)

    return {
        "bvp": recon_bvp.detach().cpu().numpy().astype(np.float32),
        "acc": recon_acc.detach().cpu().numpy().astype(np.float32),
        "slow": recon_slow.detach().cpu().numpy().astype(np.float32),
    }


# ============================================================
# Plot helpers
# ============================================================

def plot_example_windows(
    X_bvp: np.ndarray,
    X_acc: np.ndarray,
    X_slow: np.ndarray,
    y_original: np.ndarray,
    method_name: str,
    idx_to_activity_label: Dict[int, int],
    figures_dir: Path,
    gen_config: GenerationConfig,
) -> None:
    if not gen_config.save_plot:
        return

    figures_dir.mkdir(parents=True, exist_ok=True)

    n = min(int(gen_config.num_example_plots), len(y_original))

    if n <= 0:
        return

    rng = np.random.default_rng(gen_config.random_seed)
    indices = rng.choice(np.arange(len(y_original)), size=n, replace=False)

    for plot_i, idx in enumerate(indices, start=1):
        fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=False)

        axes[0].plot(X_bvp[idx, :, 0], linewidth=0.8)
        axes[0].set_title("BVP 64Hz")
        axes[0].grid(alpha=0.2)

        axes[1].plot(X_acc[idx, :, 0], linewidth=0.8, label="ACC_x")
        axes[1].plot(X_acc[idx, :, 1], linewidth=0.8, label="ACC_y")
        axes[1].plot(X_acc[idx, :, 2], linewidth=0.8, label="ACC_z")
        axes[1].set_title("ACC 32Hz")
        axes[1].legend(loc="upper right")
        axes[1].grid(alpha=0.2)

        axes[2].plot(X_slow[idx, :, 0], linewidth=0.8, label="EDA")
        axes[2].plot(X_slow[idx, :, 1], linewidth=0.8, label="TEMP")
        axes[2].set_title("SLOW 4Hz: EDA/TEMP")
        axes[2].legend(loc="upper right")
        axes[2].grid(alpha=0.2)

        fig.suptitle(
            f"{method_name} synthetic window example | index={idx} | activity={int(y_original[idx])}"
        )
        fig.tight_layout(rect=[0, 0, 1, 0.95])

        path = figures_dir / f"{method_name}_example_window_{plot_i:02d}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print("Saved:", path)


# ============================================================
# Saving generated data
# ============================================================

def build_generation_metadata(
    method_name: str,
    y_encoded: np.ndarray,
    y_original: np.ndarray,
    synthetic_subjects: np.ndarray,
    subject_tokens: np.ndarray,
    subject_condition_name_by_syn_subject: Dict[str, str],
    idx_to_subject: Dict[int, str],
    train_config: KoVAEConfig,
    gen_config: GenerationConfig,
) -> pd.DataFrame:
    rows = []

    total_windows = len(y_encoded)

    for i in range(total_windows):
        syn_subject = str(synthetic_subjects[i])
        window_in_subject = i % gen_config.windows_per_subject
        subject_token = int(subject_tokens[i])

        acc_start = window_in_subject * 64
        bvp_start = window_in_subject * 128
        slow_start = window_in_subject * 8

        rows.append(
            {
                "global_index": int(i),
                "synthetic_subject": syn_subject,
                "window_in_subject": int(window_in_subject),
                "activity_label": int(y_original[i]),
                "activity_index": int(y_encoded[i]),
                "generation_method": method_name,
                "generation_mode": (
                    "activity_conditioned_koopman_rollout"
                    if method_name == "rollout_v1"
                    else "activity_conditioned_posterior_bank"
                ),
                "subject_condition_token": subject_token,
                "subject_condition_name": idx_to_subject.get(subject_token, "UNK"),
                "synthetic_subject_assigned_condition_name": subject_condition_name_by_syn_subject[syn_subject],
                "acc_start_sample_32hz": int(acc_start),
                "acc_end_sample_32hz": int(acc_start + train_config.acc_len),
                "bvp_start_sample_64hz": int(bvp_start),
                "bvp_end_sample_64hz": int(bvp_start + train_config.bvp_len),
                "slow_start_sample_4hz": int(slow_start),
                "slow_end_sample_4hz": int(slow_start + train_config.slow_len),
            }
        )

    return pd.DataFrame(rows)


def save_generated_dataset(
    method_name: str,
    decoded: Dict[str, np.ndarray],
    y_original: np.ndarray,
    synthetic_subjects: np.ndarray,
    metadata: pd.DataFrame,
    method_paths: Dict[str, Path],
) -> None:
    make_dirs(method_paths["synthetic"], method_paths["results"], method_paths["figures"])

    np.save(method_paths["synthetic"] / "generated_subjects_X_acc_32hz.npy", decoded["acc"].astype(np.float32))
    np.save(method_paths["synthetic"] / "generated_subjects_X_bvp_64hz.npy", decoded["bvp"].astype(np.float32))
    np.save(method_paths["synthetic"] / "generated_subjects_X_slow_4hz.npy", decoded["slow"].astype(np.float32))
    np.save(method_paths["synthetic"] / "generated_subjects_all_y.npy", y_original.astype(np.int64))
    np.save(method_paths["synthetic"] / "generated_subjects_all_subject.npy", synthetic_subjects.astype(object))

    metadata.to_csv(method_paths["synthetic"] / "generated_subjects_metadata.csv", index=False)

    print(f"\nSaved synthetic data for method: {method_name}")
    print("Folder:", method_paths["synthetic"])
    print("ACC:", decoded["acc"].shape)
    print("BVP:", decoded["bvp"].shape)
    print("SLOW:", decoded["slow"].shape)
    print("y:", y_original.shape)
    print("subjects:", synthetic_subjects.shape)


def summarize_generated_dataset(
    method_name: str,
    decoded: Dict[str, np.ndarray],
    y_encoded: np.ndarray,
    y_original: np.ndarray,
    synthetic_subjects: np.ndarray,
    subject_condition_map: Dict[str, str],
    method_paths: Dict[str, Path],
    stability_df: pd.DataFrame,
    gen_config: GenerationConfig,
) -> Dict:
    activity_counts = {
        str(int(label)): int(count)
        for label, count in zip(*np.unique(y_original, return_counts=True))
    }

    subject_counts = {
        str(subject): int(count)
        for subject, count in zip(*np.unique(synthetic_subjects.astype(str), return_counts=True))
    }

    summary = {
        "method": method_name,
        "num_generated_windows": int(len(y_original)),
        "num_synthetic_subjects": int(len(np.unique(synthetic_subjects.astype(str)))),
        "windows_per_subject": int(gen_config.windows_per_subject),
        "X_acc_shape": list(decoded["acc"].shape),
        "X_bvp_shape": list(decoded["bvp"].shape),
        "X_slow_shape": list(decoded["slow"].shape),
        "activity_counts_original_labels": activity_counts,
        "subject_counts": subject_counts,
        "activity_sampling_mode": gen_config.activity_sampling_mode,
        "synthetic_subject_condition_mode": gen_config.synthetic_subject_condition_mode,
        "subject_condition_map": subject_condition_map,
        "target_spectral_radius": float(gen_config.target_spectral_radius),
        "synthetic_output_folder": str(method_paths["synthetic"]),
        "uses_quality_filter": False,
    }

    if method_name == "rollout_v1":
        summary["generation_parameters"] = {
            "rollout_z0_noise_scale": float(gen_config.rollout_z0_noise_scale),
            "rollout_latent_noise_scale": float(gen_config.rollout_latent_noise_scale),
        }

    if method_name == "posterior_bank_v2":
        summary["generation_parameters"] = {
            "posterior_noise_scale": float(gen_config.posterior_noise_scale),
            "posterior_interpolation_prob": float(gen_config.posterior_interpolation_prob),
            "posterior_koopman_blend_weight": float(gen_config.posterior_koopman_blend_weight),
        }

    save_json(summary, method_paths["results"] / "generation_summary.json")
    stability_df.to_csv(method_paths["results"] / "activity_koopman_generation_stability.csv", index=False)

    return summary


# ============================================================
# Main generation function
# ============================================================

def generate_one_method(
    method_name: str,
    model: MultiBranchKoVAE,
    train_config: KoVAEConfig,
    gen_config: GenerationConfig,
    arrays: Dict[str, np.ndarray],
    y_train_encoded: np.ndarray,
    latent_banks: Dict[str, Dict[int, np.ndarray]],
    activity_A_stable: np.ndarray,
    stability_df: pd.DataFrame,
    activity_to_idx: Dict[str, int],
    subject_to_idx: Dict[str, int],
    idx_to_activity_label: Dict[int, int],
    idx_to_subject: Dict[int, str],
    device: torch.device,
    base_paths: Dict[str, Path],
) -> Dict:
    if method_name not in {"rollout_v1", "posterior_bank_v2"}:
        raise ValueError(f"Unknown method: {method_name}")

    method_paths = get_method_paths(base_paths, method_name)
    make_dirs(method_paths["synthetic"], method_paths["results"], method_paths["figures"])

    rng = np.random.default_rng(gen_config.random_seed + abs(hash(method_name)) % 10000)

    total_windows = int(gen_config.num_synthetic_subjects * gen_config.windows_per_subject)

    y_encoded_generated = create_synthetic_activity_sequence(
        num_windows=total_windows,
        y_train_encoded=y_train_encoded,
        num_activities=len(activity_to_idx),
        gen_config=gen_config,
        rng=rng,
    )

    y_original_generated = np.asarray(
        [idx_to_activity_label[int(idx)] for idx in y_encoded_generated],
        dtype=np.int64,
    )

    synthetic_subjects = create_synthetic_subject_names(gen_config)

    subject_tokens, subject_condition_map = assign_subject_condition_tokens(
        synthetic_subjects=synthetic_subjects,
        subject_to_idx=subject_to_idx,
        gen_config=gen_config,
        rng=rng,
    )

    decoded_acc = []
    decoded_bvp = []
    decoded_slow = []

    print("\n" + "=" * 100)
    print(f"Generating method: {method_name}")
    print("=" * 100)
    print("Total windows:", total_windows)
    print("Output folder:", method_paths["synthetic"])

    for start in range(0, total_windows, gen_config.generation_batch_size):
        end = min(start + gen_config.generation_batch_size, total_windows)

        activity_batch = y_encoded_generated[start:end]
        subject_batch = subject_tokens[start:end]

        if method_name == "rollout_v1":
            z_np = generate_latent_rollout_batch(
                activity_batch=activity_batch,
                z0_banks=latent_banks["z0_banks"],
                activity_A_stable=activity_A_stable,
                train_config=train_config,
                gen_config=gen_config,
                rng=rng,
            )
        else:
            z_np = generate_posterior_bank_batch(
                activity_batch=activity_batch,
                trajectory_banks=latent_banks["trajectory_banks"],
                activity_A_stable=activity_A_stable,
                gen_config=gen_config,
                rng=rng,
            )

        decoded = decode_latent_batch(
            model=model,
            z_np=z_np,
            activity_batch=activity_batch,
            subject_token_batch=subject_batch,
            device=device,
        )

        decoded_bvp.append(decoded["bvp"])
        decoded_acc.append(decoded["acc"])
        decoded_slow.append(decoded["slow"])

        if end == total_windows or start == 0 or (start // gen_config.generation_batch_size) % 20 == 0:
            print(f"{method_name}: generated {end}/{total_windows}")

    decoded_full = {
        "bvp": np.concatenate(decoded_bvp, axis=0).astype(np.float32),
        "acc": np.concatenate(decoded_acc, axis=0).astype(np.float32),
        "slow": np.concatenate(decoded_slow, axis=0).astype(np.float32),
    }

    metadata = build_generation_metadata(
        method_name=method_name,
        y_encoded=y_encoded_generated,
        y_original=y_original_generated,
        synthetic_subjects=synthetic_subjects,
        subject_tokens=subject_tokens,
        subject_condition_name_by_syn_subject=subject_condition_map,
        idx_to_subject=idx_to_subject,
        train_config=train_config,
        gen_config=gen_config,
    )

    save_generated_dataset(
        method_name=method_name,
        decoded=decoded_full,
        y_original=y_original_generated,
        synthetic_subjects=synthetic_subjects,
        metadata=metadata,
        method_paths=method_paths,
    )

    plot_example_windows(
        X_bvp=decoded_full["bvp"],
        X_acc=decoded_full["acc"],
        X_slow=decoded_full["slow"],
        y_original=y_original_generated,
        method_name=method_name,
        idx_to_activity_label=idx_to_activity_label,
        figures_dir=method_paths["figures"],
        gen_config=gen_config,
    )

    summary = summarize_generated_dataset(
        method_name=method_name,
        decoded=decoded_full,
        y_encoded=y_encoded_generated,
        y_original=y_original_generated,
        synthetic_subjects=synthetic_subjects,
        subject_condition_map=subject_condition_map,
        method_paths=method_paths,
        stability_df=stability_df,
        gen_config=gen_config,
    )

    return summary


def run_generation(gen_config: GenerationConfig = GEN_CONFIG) -> Dict[str, Dict]:
    set_random_seed(gen_config.random_seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    model, train_config, activity_to_idx, subject_to_idx, base_paths = load_model_and_context(
        gen_config=gen_config,
        device=device,
    )

    make_dirs(
        base_paths["synthetic_base"],
        base_paths["results_base"],
        base_paths["figures_base"],
        base_paths["configs"],
    )

    save_json(asdict(gen_config), base_paths["configs"] / "kovae_generation_config.json")

    arrays = load_native_rate_arrays(base_paths["processed"])
    validate_arrays(arrays, train_config)

    y_encoded_all = encode_activities(arrays["y"], activity_to_idx)
    subject_encoded_all = encode_subjects(arrays["subjects"], subject_to_idx)

    train_indices = build_indices_by_subject(arrays["subjects"], train_config.train_subjects)

    if len(train_indices) == 0:
        raise RuntimeError("No train windows found. Check train_subjects.")

    y_train_encoded = y_encoded_all[train_indices]

    idx_to_activity_label = make_idx_to_activity_label(activity_to_idx)
    idx_to_subject = make_idx_to_subject(subject_to_idx)

    print("\nActivity mapping:", activity_to_idx)
    print("Subject mapping:", subject_to_idx)
    print("Train windows used for latent banks:", len(train_indices))

    latent_banks = collect_activity_latent_banks(
        model=model,
        arrays=arrays,
        y_encoded=y_encoded_all,
        subject_encoded=subject_encoded_all,
        train_indices=train_indices,
        batch_size=gen_config.generation_batch_size,
        device=device,
    )

    save_latent_banks(latent_banks, base_paths["results_base"])

    activity_A_raw = model.get_activity_koopman_matrices().detach().cpu().numpy().astype(np.float32)

    activity_A_stable, stability_df = stabilize_activity_koopman_matrices(
        activity_A_raw=activity_A_raw,
        target_radius=gen_config.target_spectral_radius,
    )

    np.save(
        base_paths["results_base"] / "activity_koopman_matrices_raw.npy",
        activity_A_raw.astype(np.float32),
    )
    np.save(
        base_paths["results_base"] / "activity_koopman_matrices_stable_for_generation.npy",
        activity_A_stable.astype(np.float32),
    )
    stability_df.to_csv(
        base_paths["results_base"] / "activity_koopman_generation_stability.csv",
        index=False,
    )

    print("\nActivity-conditioned Koopman stability:")
    print(stability_df)

    method_summaries = {}

    for method_name in gen_config.methods_to_run:
        method_summaries[method_name] = generate_one_method(
            method_name=method_name,
            model=model,
            train_config=train_config,
            gen_config=gen_config,
            arrays=arrays,
            y_train_encoded=y_train_encoded,
            latent_banks=latent_banks,
            activity_A_stable=activity_A_stable,
            stability_df=stability_df,
            activity_to_idx=activity_to_idx,
            subject_to_idx=subject_to_idx,
            idx_to_activity_label=idx_to_activity_label,
            idx_to_subject=idx_to_subject,
            device=device,
            base_paths=base_paths,
        )

    combined_summary = {
        "methods_run": gen_config.methods_to_run,
        "checkpoint": str(base_paths["checkpoint"]),
        "synthetic_base": str(base_paths["synthetic_base"]),
        "results_base": str(base_paths["results_base"]),
        "figures_base": str(base_paths["figures_base"]),
        "uses_activity_conditioned_koopman": True,
        "uses_quality_filter": False,
        "method_summaries": method_summaries,
    }

    save_json(combined_summary, base_paths["results_base"] / "combined_generation_summary.json")

    print("\n" + "#" * 100)
    print("Synthetic generation finished.")
    print("#" * 100)
    print("Combined summary:", base_paths["results_base"] / "combined_generation_summary.json")

    return combined_summary


if __name__ == "__main__":
    outputs = run_generation(GEN_CONFIG)


Using device: cuda

Activity mapping: {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7}
Subject mapping: {'UNK': 0, 'S1': 1, 'S2': 2, 'S3': 3, 'S4': 4, 'S5': 5, 'S6': 6, 'S9': 7, 'S11': 8, 'S12': 9, 'S13': 10}
Train windows used for latent banks: 30762

Latent bank sizes:
activity_idx=0 | z0=(3032, 32) | trajectory=(3032, 64, 32)
activity_idx=1 | z0=(2139, 32) | trajectory=(2139, 64, 32)
activity_idx=2 | z0=(1500, 32) | trajectory=(1500, 64, 32)
activity_idx=3 | z0=(2296, 32) | trajectory=(2296, 64, 32)
activity_idx=4 | z0=(4580, 32) | trajectory=(4580, 64, 32)
activity_idx=5 | z0=(8792, 32) | trajectory=(8792, 64, 32)
activity_idx=6 | z0=(2903, 32) | trajectory=(2903, 64, 32)
activity_idx=7 | z0=(5520, 32) | trajectory=(5520, 64, 32)

Activity-conditioned Koopman stability:
   activity_idx  raw_spectral_radius  target_spectral_radius  scaling_factor  \
0             0             1.451715                    0.98        0.675064   
1             1             1.393109   

## Final run cell

Run this after Notebook 02 has trained the new main KoVAE and saved:

```text
models/checkpoints/kovae_best.pt
```

Default final generation:

```python
GEN_CONFIG.methods_to_run = ["rollout_v1", "posterior_bank_v2"]
GEN_CONFIG.num_synthetic_subjects = 10
GEN_CONFIG.windows_per_subject = 3000
```

No filter is used by default, because the updated KoVAE already has activity-conditioned training.


In [2]:
GEN_CONFIG.methods_to_run = ["rollout_v1", "posterior_bank_v2"]

GEN_CONFIG.num_synthetic_subjects = 10
GEN_CONFIG.windows_per_subject = 3000
GEN_CONFIG.generation_batch_size = 128

# For final data, keep the real train activity distribution.
GEN_CONFIG.activity_sampling_mode = "train_distribution"

# Gives subject-style variation using train-subject tokens only.
# Use "UNK" if you want a more generic subject condition.
GEN_CONFIG.synthetic_subject_condition_mode = "sample_train_subject"

GEN_CONFIG.target_spectral_radius = 0.98

GEN_CONFIG.rollout_z0_noise_scale = 0.05
GEN_CONFIG.rollout_latent_noise_scale = 0.03

GEN_CONFIG.posterior_noise_scale = 0.04
GEN_CONFIG.posterior_interpolation_prob = 0.50
GEN_CONFIG.posterior_koopman_blend_weight = 0.10

GEN_CONFIG.save_plot = True

outputs = run_generation(GEN_CONFIG)
outputs


Using device: cuda

Activity mapping: {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7}
Subject mapping: {'UNK': 0, 'S1': 1, 'S2': 2, 'S3': 3, 'S4': 4, 'S5': 5, 'S6': 6, 'S9': 7, 'S11': 8, 'S12': 9, 'S13': 10}
Train windows used for latent banks: 30762

Latent bank sizes:
activity_idx=0 | z0=(3032, 32) | trajectory=(3032, 64, 32)
activity_idx=1 | z0=(2139, 32) | trajectory=(2139, 64, 32)
activity_idx=2 | z0=(1500, 32) | trajectory=(1500, 64, 32)
activity_idx=3 | z0=(2296, 32) | trajectory=(2296, 64, 32)
activity_idx=4 | z0=(4580, 32) | trajectory=(4580, 64, 32)
activity_idx=5 | z0=(8792, 32) | trajectory=(8792, 64, 32)
activity_idx=6 | z0=(2903, 32) | trajectory=(2903, 64, 32)
activity_idx=7 | z0=(5520, 32) | trajectory=(5520, 64, 32)

Activity-conditioned Koopman stability:
   activity_idx  raw_spectral_radius  target_spectral_radius  scaling_factor  \
0             0             1.451715                    0.98        0.675064   
1             1             1.393109   

{'methods_run': ['rollout_v1', 'posterior_bank_v2'],
 'checkpoint': '/home/iailab42/khans1/projects/ir/models/checkpoints/kovae_best.pt',
 'synthetic_base': '/home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae',
 'results_base': '/home/iailab42/khans1/projects/ir/results/kovae_generation',
 'figures_base': '/home/iailab42/khans1/projects/ir/figures/kovae_generation',
 'uses_activity_conditioned_koopman': True,
 'uses_quality_filter': False,
 'method_summaries': {'rollout_v1': {'method': 'rollout_v1',
   'num_generated_windows': 30000,
   'num_synthetic_subjects': 10,
   'windows_per_subject': 3000,
   'X_acc_shape': [30000, 256, 3],
   'X_bvp_shape': [30000, 512, 1],
   'X_slow_shape': [30000, 32, 2],
   'activity_counts_original_labels': {'1': 2900,
    '2': 2051,
    '3': 1397,
    '4': 2190,
    '5': 4616,
    '6': 8628,
    '7': 2756,
    '8': 5462},
   'subject_counts': {'synthetic_subject_01': 3000,
    'synthetic_subject_02': 3000,
    'synthetic_subject_03': 3000,
 